# Tayara Real Estate — ML-Ready Data Cleaning

This notebook takes the `clean_tayara` table and produces a fully ML-ready DataFrame.

**Steps:**
1. Load data from PostgreSQL
2. Remove garbage / impossible values (negative prices, extreme outliers)
3. Standardize categorical columns (transaction type, property type, usage)
4. Consolidate duplicate/redundant columns
5. Fill or drop missing values with ML-appropriate strategy
6. Feature engineering (log-price, age of listing, geo precision flag)
7. Final column selection & type casting
8. Export to Parquet

## 1. Load Data

In [1]:
import pandas as pd
import numpy as np
import psycopg2
import os
import warnings
warnings.filterwarnings('ignore')

DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_USER = 'airflow'
DB_PASSWORD = 'airflow'
DB_NAME = 'airflow'

conn = psycopg2.connect(host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD, database=DB_NAME)
df = pd.read_sql_query('SELECT * FROM clean_tayara', conn)
conn.close()

print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")

Loaded: 9227 rows × 37 columns


## 2. Drop Useless / Near-Empty Columns

Columns dropped and why:
- `adresse_raw`, `quartier`, `delegation`, `gouvernorat` → >89% missing; replaced by matched geo columns
- `num_etage` → 86% missing; niche feature, can be re-added later
- `titre_foncier_extrait` → 94% missing
- `scraped_at` → same-day scrape, no signal
- `location`, `transaction_type`, `superficie_num`, `nbr_chambres_num`, `nbr_sdb_num` → raw pre-ETL versions superseded by `_final` columns
- `geo_source` → single value (no signal)
- `id` → identifier, not a feature

In [2]:
DROP_COLS = [
    # identifiers
    'id',
    # superseded raw columns
    'location', 'transaction_type', 'superficie_num', 'nbr_chambres_num', 'nbr_sdb_num',
    # near-empty geo fields
    'adresse_raw', 'quartier', 'delegation', 'gouvernorat',
    # near-empty others
    'num_etage', 'titre_foncier_extrait',
    # no ML signal
    'scraped_at', 'geo_source',
]

df = df.drop(columns=DROP_COLS)
print(f"After drop: {df.shape[1]} columns remaining")
print(df.columns.tolist())

After drop: 23 columns remaining
['title', 'location_finale', 'transaction_type_final', 'price_num', 'superficie_finale', 'nbr_chambres_final', 'nbr_sdb_final', 'image_count', 'pub_hours_ago', 'price_per_m2', 'published_at_from_id', 'ville', 'matched_state', 'matched_delegation', 'matched_locality_name', 'matched_postal_code', 'latitude', 'longitude', 'geo_match_level', 'type_bien_extrait', 'usage_extrait', 'source_extraction', 'extraction_confidence']


## 3. Fix Impossible / Garbage Numeric Values

Issues spotted in EDA:
- `price_num` has negatives and a max of 11 trillion — clear scraping artifacts
- `superficie_finale` has negatives and values like 58 million m²
- `nbr_chambres_final` / `nbr_sdb_final` have negatives and 1-million+ values
- `price_per_m2` has negatives (derived from bad price/superficie)

Strategy: set implausible values to NaN, then handle in the missing-value step.

In [3]:
# ── Price ──────────────────────────────────────────────────────────────────
# Tunisian real estate: reasonable range 100 TND (rent) to ~50 M TND (luxury)
PRICE_MIN = 100
PRICE_MAX = 50_000_000
df.loc[~df['price_num'].between(PRICE_MIN, PRICE_MAX), 'price_num'] = np.nan
print(f"price_num NaN after fix: {df['price_num'].isna().sum()}")

# ── Superficie ─────────────────────────────────────────────────────────────
# Reasonable: 5 m² (studio) to 500,000 m² (large farm / terrain)
SURF_MIN = 5
SURF_MAX = 500_000
df.loc[~df['superficie_finale'].between(SURF_MIN, SURF_MAX), 'superficie_finale'] = np.nan
print(f"superficie_finale NaN after fix: {df['superficie_finale'].isna().sum()}")

# ── Rooms ──────────────────────────────────────────────────────────────────
ROOM_MIN = 0
ROOM_MAX = 50   # 50 rooms is already a mansion/commercial building
for col in ['nbr_chambres_final', 'nbr_sdb_final']:
    df.loc[~df[col].between(ROOM_MIN, ROOM_MAX), col] = np.nan
    print(f"{col} NaN after fix: {df[col].isna().sum()}")

# ── Recompute price_per_m2 from clean values ────────────────────────────────
df['price_per_m2'] = np.where(
    df['price_num'].notna() & df['superficie_finale'].notna() & (df['superficie_finale'] > 0),
    df['price_num'] / df['superficie_finale'],
    np.nan
)
print(f"price_per_m2 NaN after recompute: {df['price_per_m2'].isna().sum()}")

price_num NaN after fix: 1707
superficie_finale NaN after fix: 1813
nbr_chambres_final NaN after fix: 2757
nbr_sdb_final NaN after fix: 2316
price_per_m2 NaN after recompute: 3029


## 4. Standardize Categorical Columns

- `transaction_type_final`: 83 unique values → collapse to 3 clean categories
- `type_bien_extrait`: 210 unique values (case variants, Arabic/French mix) → normalize
- `usage_extrait`: 58 unique values → normalize
- `geo_match_level`: already clean (3 values)
- `extraction_confidence` / `source_extraction`: keep as-is

In [4]:
# ── Helper ─────────────────────────────────────────────────────────────────
def normalize_str(s):
    if pd.isna(s):
        return np.nan
    return str(s).strip().lower()

# ── Transaction type ────────────────────────────────────────────────────────
SALE_KEYWORDS  = ['vendre', 'vente', 'sale', 'بيع', 'a vendre', 'à vendre']
RENT_KEYWORDS  = ['louer', 'location', 'rent', 'إيجار', 'كراء', 'à louer']

def map_transaction(val):
    v = normalize_str(val)
    if pd.isna(v): return np.nan
    if any(k in v for k in SALE_KEYWORDS): return 'vente'
    if any(k in v for k in RENT_KEYWORDS): return 'location'
    return np.nan  # exchange / unclear → treat as unknown

df['transaction'] = df['transaction_type_final'].apply(map_transaction)
print("transaction:\n", df['transaction'].value_counts(dropna=False))
df = df.drop(columns=['transaction_type_final'])

# ── Property type ───────────────────────────────────────────────────────────
TYPE_MAP = {
    # Appartement
    'appartement': 'appartement', 'appart': 'appartement', 'apartment': 'appartement',
    'studio': 'appartement', 's1': 'appartement', 's2': 'appartement',
    's3': 'appartement', 's4': 'appartement', 's+1': 'appartement',
    's+2': 'appartement', 's+3': 'appartement', 's+4': 'appartement',
    # Villa / Maison
    'villa': 'villa', 'maison': 'villa', 'duplex': 'villa',
    'triplex': 'villa', 'ferme': 'villa',
    # Terrain
    'terrain': 'terrain',
    # Bureau
    'bureau': 'bureau', 'office': 'bureau', 'open space': 'bureau',
    # Commerce
    'local commercial': 'commerce', 'commerce': 'commerce', 'magasin': 'commerce',
    'boutique': 'commerce', 'restaurant': 'commerce', 'hotel': 'commerce',
    # Immeuble
    'immeuble': 'immeuble', 'building': 'immeuble',
    # Garage / Parking
    'garage': 'garage', 'parking': 'garage',
}

def map_type_bien(val):
    v = normalize_str(val)
    if pd.isna(v): return np.nan
    for k, mapped in TYPE_MAP.items():
        if k in v: return mapped
    return 'autre'

df['type_bien'] = df['type_bien_extrait'].apply(map_type_bien)
print("\ntype_bien:\n", df['type_bien'].value_counts(dropna=False))
df = df.drop(columns=['type_bien_extrait'])

# ── Usage ────────────────────────────────────────────────────────────────────
USAGE_MAP = {
    'habitation': 'habitation', 'résidentiel': 'habitation', 'residential': 'habitation',
    'meublé': 'habitation',
    'terrain': 'terrain',
    'bureau': 'bureau',
    'commerce': 'commerce', 'commercial': 'commerce',
    'industriel': 'industriel', 'industrie': 'industriel',
    'agricole': 'agricole', 'ferme': 'agricole',
}

def map_usage(val):
    v = normalize_str(val)
    if pd.isna(v): return np.nan
    for k, mapped in USAGE_MAP.items():
        if k in v: return mapped
    return 'autre'

df['usage'] = df['usage_extrait'].apply(map_usage)
print("\nusage:\n", df['usage'].value_counts(dropna=False))
df = df.drop(columns=['usage_extrait'])

transaction:
 transaction
location    4565
vente       4020
NaN          642
Name: count, dtype: int64

type_bien:
 type_bien
appartement    3583
villa          1932
terrain        1289
bureau          900
autre           876
NaN             409
commerce        124
immeuble        103
garage           11
Name: count, dtype: int64

usage:
 usage
habitation    3819
NaN           2015
terrain       1708
bureau        1089
commerce       511
autre           76
agricole         5
industriel       4
Name: count, dtype: int64


## 5. Geography — Pick Best Available Level

In [5]:
# Use matched_state (gouvernorat) as primary geographic feature — only 1.6% missing
# matched_delegation available for 43% — keep as secondary feature
# geo_match_level: encode precision

GEO_PRECISION = {'none': 0, 'state': 1, 'delegation': 2}
df['geo_precision'] = df['geo_match_level'].map(GEO_PRECISION).fillna(0).astype(int)
df = df.drop(columns=['geo_match_level'])

# Rename for clarity
df = df.rename(columns={
    'location_finale': 'governorate_raw',   # original Tayara label
    'matched_state': 'gouvernorat',          # clean matched governorate
    'matched_delegation': 'delegation',
    'matched_locality_name': 'locality',
    'matched_postal_code': 'postal_code',
})

# Drop raw governorate (superseded by matched)
df = df.drop(columns=['governorate_raw', 'ville'])

print("Governorat missing:", df['gouvernorat'].isna().sum())
print("Delegation missing:", df['delegation'].isna().sum())
print("Lat/Lon missing:", df['latitude'].isna().sum())

Governorat missing: 152
Delegation missing: 5262
Lat/Lon missing: 5243


## 6. Handle Missing Values

In [6]:
# ── Drop rows where price is missing (target variable for most ML tasks) ───
before = len(df)
df = df.dropna(subset=['price_num'])
print(f"Dropped {before - len(df)} rows with missing price. Remaining: {len(df)}")

# ── Drop rows where gouvernorat is missing (primary location feature) ───────
before = len(df)
df = df.dropna(subset=['gouvernorat'])
print(f"Dropped {before - len(df)} rows with missing gouvernorat. Remaining: {len(df)}")

# ── Drop rows with unknown transaction type ─────────────────────────────────
before = len(df)
df = df.dropna(subset=['transaction'])
print(f"Dropped {before - len(df)} rows with unknown transaction. Remaining: {len(df)}")

# ── Numeric: median imputation per gouvernorat+type_bien group ──────────────
for col in ['superficie_finale', 'nbr_chambres_final', 'nbr_sdb_final']:
    group_medians = df.groupby(['gouvernorat', 'type_bien'])[col].transform('median')
    global_median = df[col].median()
    df[col] = df[col].fillna(group_medians).fillna(global_median)
    print(f"{col} — remaining NaN: {df[col].isna().sum()}")

# ── price_per_m2: recompute after imputation ────────────────────────────────
df['price_per_m2'] = df['price_num'] / df['superficie_finale'].replace(0, np.nan)

# ── Categorical: fill with 'inconnu' ────────────────────────────────────────
for col in ['type_bien', 'usage', 'delegation', 'locality', 'postal_code']:
    df[col] = df[col].fillna('inconnu')

# ── Geo coords: fill with governorate centroid ──────────────────────────────
gov_lat = df.groupby('gouvernorat')['latitude'].transform('median')
gov_lon = df.groupby('gouvernorat')['longitude'].transform('median')
df['latitude']  = df['latitude'].fillna(gov_lat)
df['longitude'] = df['longitude'].fillna(gov_lon)
df['geo_imputed'] = (df['latitude'].isna() | df['geo_precision'] < 2).astype(int)

# ── pub_hours_ago: fill with median ─────────────────────────────────────────
df['pub_hours_ago'] = df['pub_hours_ago'].fillna(df['pub_hours_ago'].median())

print(f"\nFinal missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

Dropped 1707 rows with missing price. Remaining: 7520
Dropped 122 rows with missing gouvernorat. Remaining: 7398
Dropped 477 rows with unknown transaction. Remaining: 6921
superficie_finale — remaining NaN: 0
nbr_chambres_final — remaining NaN: 0
nbr_sdb_final — remaining NaN: 0

Final missing values:
latitude     19
longitude    19
dtype: int64


## 7. Feature Engineering

In [7]:
# ── Log-price (better distribution for regression models) ───────────────────
df['log_price'] = np.log1p(df['price_num'])

# ── Log-superficie ──────────────────────────────────────────────────────────
df['log_superficie'] = np.log1p(df['superficie_finale'])

# ── Listing age in days (from publication date to scrape date) ───────────────
df['published_at'] = pd.to_datetime(df['published_at_from_id'], utc=True, errors='coerce')
SCRAPE_DATE = pd.Timestamp('2026-04-09', tz='UTC')
df['listing_age_days'] = (SCRAPE_DATE - df['published_at']).dt.days
df['listing_age_days'] = df['listing_age_days'].clip(lower=0)
df = df.drop(columns=['published_at_from_id', 'published_at'])

# ── Rooms ratio (bathrooms / bedrooms) ─────────────────────────────────────
df['sdb_per_chambre'] = np.where(
    df['nbr_chambres_final'] > 0,
    df['nbr_sdb_final'] / df['nbr_chambres_final'],
    0
)

# ── Extraction confidence as binary: high vs normal ─────────────────────────
df['high_confidence'] = (df['extraction_confidence'] >= 0.8).astype(int)

# ── Is rent vs sale (binary) ────────────────────────────────────────────────
df['is_location'] = (df['transaction'] == 'location').astype(int)

print("New features added: log_price, log_superficie, listing_age_days, sdb_per_chambre, high_confidence, is_location")
print(f"Shape: {df.shape}")

New features added: log_price, log_superficie, listing_age_days, sdb_per_chambre, high_confidence, is_location
Shape: (6921, 27)


## 8. Final Column Selection & Types

In [8]:
# Cast categoricals
CAT_COLS = [
    'transaction', 'type_bien', 'usage',
    'gouvernorat', 'delegation', 'locality', 'postal_code',
    'geo_match_level_str',  # we'll add this
    'source_extraction',
]

# Recreate geo_match_level as string label from geo_precision
df['geo_match_level_str'] = df['geo_precision'].map({0: 'none', 1: 'state', 2: 'delegation'})

for col in CAT_COLS:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Final selected columns (ordered logically)
ML_COLUMNS = [
    # Target
    'price_num', 'log_price',
    # Transaction
    'transaction', 'is_location',
    # Property
    'type_bien', 'usage',
    'superficie_finale', 'log_superficie',
    'nbr_chambres_final', 'nbr_sdb_final', 'sdb_per_chambre',
    'price_per_m2',
    # Listing metadata
    'image_count', 'pub_hours_ago', 'listing_age_days',
    # Geography
    'gouvernorat', 'delegation', 'locality', 'postal_code',
    'latitude', 'longitude',
    'geo_precision', 'geo_match_level_str', 'geo_imputed',
    # Extraction quality
    'source_extraction', 'extraction_confidence', 'high_confidence',
    # Title (for NLP-based models)
    'title',
]

df_ml = df[ML_COLUMNS].copy()

print(f"Final ML dataset: {df_ml.shape}")
print(f"\nMissing values:")
miss = df_ml.isnull().sum()
print(miss[miss > 0] if miss.sum() > 0 else "  None!")

Final ML dataset: (6921, 28)

Missing values:
latitude     19
longitude    19
dtype: int64


## 9. Quick Sanity Checks

In [9]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Rows: {len(df_ml):,}")
print(f"Columns: {df_ml.shape[1]}")

print("\n--- Transaction split ---")
print(df_ml['transaction'].value_counts())

print("\n--- Property type ---")
print(df_ml['type_bien'].value_counts())

print("\n--- Price (TND) ---")
print(df_ml['price_num'].describe().apply(lambda x: f"{x:,.0f}"))

print("\n--- Top governorats ---")
print(df_ml['gouvernorat'].value_counts().head(10))

print("\n--- Geo precision ---")
print(df_ml['geo_match_level_str'].value_counts())

print("\n--- Listing age (days) ---")
print(df_ml['listing_age_days'].describe())

DATASET SUMMARY
Rows: 6,921
Columns: 28

--- Transaction split ---
transaction
location    3730
vente       3191
Name: count, dtype: int64

--- Property type ---
type_bien
appartement    2967
villa          1455
bureau          810
terrain         711
autre           573
inconnu         235
commerce         84
immeuble         78
garage            8
Name: count, dtype: int64

--- Price (TND) ---
count         6,921
mean        212,793
std         919,234
min             100
25%             950
50%           2,500
75%         220,000
max      42,000,000
Name: price_num, dtype: str

--- Top governorats ---
gouvernorat
TUNIS        2294
ARIANA       1100
SOUSSE        830
BEN AROUS     699
NABEUL        684
BIZERTE       426
SFAX          283
ZAGHOUAN      186
MAHDIA        113
MONASTIR       99
Name: count, dtype: int64

--- Geo precision ---
geo_match_level_str
state         3885
delegation    3036
Name: count, dtype: int64

--- Listing age (days) ---
count    6921.000000
mean       63.

## 10. Export

In [ ]:
from pathlib import Path

out = Path('data_exports')
out.mkdir(exist_ok=True)

# Parquet — best for ML pipelines (preserves dtypes, fast)
df_ml.to_parquet(out / 'tayara_ml_ready.parquet', index=False)
print(f"✓ Saved parquet: {out / 'tayara_ml_ready.parquet'}")

# CSV — for inspection / sharing
df_ml.to_csv(out / 'tayara_ml_ready.csv', index=False)
print(f"✓ Saved CSV: {out / 'tayara_ml_ready.csv'}")

print(f"\nFinal shape: {df_ml.shape}")

## What's Next

### For Price Prediction (Regression)
```python
# Recommended features
FEATURES = [
    'log_superficie', 'nbr_chambres_final', 'nbr_sdb_final',
    'gouvernorat', 'type_bien', 'usage', 'transaction',
    'latitude', 'longitude', 'geo_precision',
    'image_count', 'listing_age_days'
]
TARGET = 'log_price'  # remember to np.expm1() predictions back to TND
```

### Encode Categoricals
```python
# Option A — LightGBM / XGBoost: pass as category dtype directly
# Option B — Sklearn: use OrdinalEncoder or TargetEncoder for high-cardinality cols
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
```

### Filter by Task
```python
# Sale only (price prediction makes more sense)
df_sale = df_ml[df_ml['transaction'] == 'vente']
# Residential only
df_res = df_ml[df_ml['usage'] == 'habitation']
```

### NLP on `title`
```python
# Titles mix French and Arabic — consider multilingual embeddings:
# CAMeL-BERT (Arabic), or sentence-transformers 'paraphrase-multilingual-mpnet-base-v2'
```